# Composio (Python HTTP client)

This notebook uses the **`composio-client`** package — the typed OpenAPI client Composio ships for REST calls. The umbrella **`composio`** PyPI package is **not** used here because it pulls in native extensions (e.g. via `openai` → `jiter`) that **do not install in Pyodide**.

## What you'll learn

1. Install **`attrs`** then **`composio-client`** with `micropip` (avoids wrong PyPI `attr` package)
2. Set your **API key** — enough for **listing toolkits and tools**
3. Optionally set **user id** + **connected account id** so **`tools.execute`** can run
4. Plot a small toolkit summary chart

### Credentials (read before running)

**API key only:** `toolkits.list` and `tools.list` work once you replace the API key placeholder — no OAuth required for those catalog calls.

**tools.execute:** Composio returns **400** unless you pass **`user_id`** (your stable end-user id in your app) **and** **`connected_account_id`** from the dashboard after you link an app (e.g. GitHub). Leave those placeholders to skip execute while you explore lists.

- Edit the **next code cell**; replace placeholders at runtime only.
- **Do not** save or commit the notebook after pasting real secrets.
- Auth is sent as **`x-api-key`** by the client (not pack KV / `proxies.yml` inject).

In [ ]:
import os

COMPOSIO_API_KEY_PLACEHOLDER = "<REPLACE_AT_RUNTIME>"

# --- Paste at runtime only (do not commit real values) ---
COMPOSIO_API_KEY = "<REPLACE_AT_RUNTIME>"
COMPOSIO_BASE_URL = "https://backend.composio.dev"

# Optional — required for tools.execute after linking an integration in the Composio UI:
COMPOSIO_USER_ID = "<REPLACE_AT_RUNTIME>"  # stable id for this end user in YOUR app (e.g. "demo-user-1")
COMPOSIO_CONNECTED_ACCOUNT_ID = "<REPLACE_AT_RUNTIME>"  # from Composio → Connected accounts

# Optional: read from process env if your host injects secrets (cell literal wins if still placeholder).
_env_key = os.environ.get("COMPOSIO_API_KEY", "").strip()
if _env_key and COMPOSIO_API_KEY == COMPOSIO_API_KEY_PLACEHOLDER:
    COMPOSIO_API_KEY = _env_key
_env_uid = os.environ.get("COMPOSIO_USER_ID", "").strip()
if _env_uid and COMPOSIO_USER_ID == COMPOSIO_API_KEY_PLACEHOLDER:
    COMPOSIO_USER_ID = _env_uid
_env_ca = os.environ.get("COMPOSIO_CONNECTED_ACCOUNT_ID", "").strip()
if _env_ca and COMPOSIO_CONNECTED_ACCOUNT_ID == COMPOSIO_API_KEY_PLACEHOLDER:
    COMPOSIO_CONNECTED_ACCOUNT_ID = _env_ca
"ready" if COMPOSIO_API_KEY != COMPOSIO_API_KEY_PLACEHOLDER else "set COMPOSIO_API_KEY (or env) to call list APIs"

### Setup

Run **once** per fresh kernel. Install **`attrs` before `composio-client`**: Pyodide’s optional auto-install maps the import name `attr` to the wrong PyPI distribution **`attr`** (tiny stub; no `attr.s`). The real module comes from **`attrs`** and prevents `AttributeError: module 'attr' has no attribute 's'` when loading the client.

In [ ]:
import micropip

# attrs first — PyPI package "attr" is unrelated; composio stack needs "attrs" for `import attr`.
await micropip.install(['attrs', 'composio-client==1.39.0'])

In [ ]:
def _is_placeholder(value: str) -> bool:
    v = (value or "").strip()
    return (not v) or (v == COMPOSIO_API_KEY_PLACEHOLDER)


def composio_credentials_ready() -> bool:
    return not _is_placeholder(COMPOSIO_API_KEY)


def composio_execute_ready() -> bool:
    return composio_credentials_ready() and not _is_placeholder(COMPOSIO_USER_ID) and not _is_placeholder(
        COMPOSIO_CONNECTED_ACCOUNT_ID
    )


client = None
if composio_credentials_ready():
    from composio_client import Composio

    client = Composio(api_key=COMPOSIO_API_KEY.strip(), base_url=COMPOSIO_BASE_URL)
    print("Composio client ready (base_url=", COMPOSIO_BASE_URL, ")")
else:
    print(
        "Skipping Composio client: set COMPOSIO_API_KEY in the credentials cell (or COMPOSIO_API_KEY env). "
        "Bundled examples use the placeholder so list calls are skipped."
    )

In [ ]:
if client is None:
    print("Skipping: toolkits.list (no client).")
else:
    try:
        page = client.toolkits.list(limit=10)
        items = page.items or []
        print(f"toolkits (up to 10): {len(items)} returned")
        for t in items[:10]:
            print(f"  - {t.slug}: {t.name}")
    except Exception as e:
        print("toolkits.list failed:", type(e).__name__, str(e)[:500])

In [ ]:
if client is None:
    print("Skipping: tools.list (no client).")
else:
    try:
        page = client.tools.list(toolkit_slug="github", limit=8)
        items = page.items or []
        print(f"github tools (up to 8): {len(items)} returned")
        for t in items[:8]:
            print(f"  - {t.slug}: {t.name}")
    except Exception as e:
        print("tools.list failed:", type(e).__name__, str(e)[:500])

## Checkpoint

With a real **API key** you should see toolkit and tool rows above. The **execute** cell runs only when **user id** and **connected account id** are also set.

In [ ]:
if client is None:
    print("Skipping: tools.execute (no client).")
elif not composio_execute_ready():
    print(
        "Skipping tools.execute: set COMPOSIO_USER_ID and COMPOSIO_CONNECTED_ACCOUNT_ID in the credentials cell "
        "(Composio dashboard → Connected accounts). API key alone is enough for list calls above."
    )
else:
    demo_slug = None
    try:
        page = client.tools.list(toolkit_slug="github", limit=25)
        for t in page.items or []:
            if getattr(t, "no_auth", False):
                demo_slug = t.slug
                break
        if demo_slug is None and (page.items or []):
            demo_slug = page.items[0].slug
    except Exception as e:
        print("Could not pick a tool slug:", type(e).__name__, str(e)[:300])
        demo_slug = None
    if not demo_slug:
        print("No tool slug selected; skipping execute.")
    else:
        try:
            out = client.tools.execute(
                demo_slug,
                user_id=COMPOSIO_USER_ID.strip(),
                connected_account_id=COMPOSIO_CONNECTED_ACCOUNT_ID.strip(),
                arguments={},
            )
            print("execute", demo_slug, "->", type(out).__name__)
            print(out)
        except Exception as e:
            print("tools.execute failed:", type(e).__name__, str(e)[:800])

In [ ]:
if client is None:
    print("Skipping chart (no client).")
else:
    try:
        import matplotlib.pyplot as plt

        page = client.toolkits.list(limit=12)
        items = page.items or []
        labels = [t.slug for t in items[:12]]
        if not labels:
            print("No toolkits to chart.")
        else:
            values = list(range(1, len(labels) + 1))
            plt.figure(figsize=(8, 4))
            plt.bar(labels, values)
            plt.title("Toolkit sample (ordinal index — demo only)")
            plt.ylabel("order")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print("chart failed:", type(e).__name__, str(e)[:400])

## Next steps

- Link an app in the [Composio](https://composio.dev) dashboard, copy **connected account id**, set **user id** to match your app user, then re-run **execute**.
- Open `Cribl_Python_SDK.ipynb` for Cribl inventory patterns side-by-side.

In [ ]:
# ### Prompt:
# Suggest the next Composio tool calls to explore based on listed toolkits.
# Context: paste the printed toolkit/tool lines from above.
# Requirements:
# - propose 3 concrete tool slugs
# - note which need OAuth vs no_auth
# - keep answers short